In [ ]:
import os, shutil

for d in ["/content/VisDrone2019-DET", "/content/VisDrone2019-DET-val", "/content/VisDrone2019-DET-train"]:
    if os.path.exists(d):
        shutil.rmtree(d)

In [ ]:
from google.colab import drive
import os, shutil, zipfile
from tqdm import tqdm

drive.mount('/content/drive', force_remount=True)

TRAIN_ZIP = "/content/drive/MyDrive/VisDrone2019-DET-train.zip"
VAL_ZIP = "/content/drive/MyDrive/VisDrone2019-DET-val.zip"
BASE_DIR = "/content/VisDrone2019-DET"

if os.path.exists(BASE_DIR):
    shutil.rmtree(BASE_DIR)

def extract_zip(zip_path, dest):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(dest)

extract_zip(TRAIN_ZIP, BASE_DIR)
extract_zip(VAL_ZIP, BASE_DIR)
print("Datasets extracted successfully.")

In [ ]:
import os, cv2
from tqdm import tqdm

TRAIN_IMG_DIR = os.path.join(BASE_DIR, "images/train")
TRAIN_ANN_DIR = os.path.join(BASE_DIR, "annotations/train")
TRAIN_LABEL_DIR = os.path.join(BASE_DIR, "labels/train")

VAL_IMG_DIR = os.path.join(BASE_DIR, "images/val")
VAL_ANN_DIR = os.path.join(BASE_DIR, "annotations/val")
VAL_LABEL_DIR = os.path.join(BASE_DIR, "labels/val")

os.makedirs(TRAIN_LABEL_DIR, exist_ok=True)
os.makedirs(VAL_LABEL_DIR, exist_ok=True)

def convert_visdrone_to_yolo(img_dir, ann_dir, label_dir):
    files = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]
    for fname in tqdm(files, desc=f"Converting {os.path.basename(img_dir)}"):
        ann_file = fname.replace('.jpg', '.txt')
        ann_path = os.path.join(ann_dir, ann_file)
        if not os.path.exists(ann_path):
            continue

        img_path = os.path.join(img_dir, fname)
        h, w, _ = cv2.imread(img_path).shape

        yolo_lines = []
        with open(ann_path, 'r') as f:
            for line in f:
                parts = list(map(float, line.strip().split(',')))
                if len(parts) < 7:
                    continue
                x, y, bw, bh, score, cls_id, occ = parts[:7]
                if occ >= 2 or cls_id == 0:
                    continue
                if cls_id not in range(1, 11):
                    continue

                cx, cy = (x + bw/2) / w, (y + bh/2) / h
                nw, nh = bw / w, bh / h
                yolo_lines.append(f"{cls_id-1} {cx} {cy} {nw} {nh}")

        with open(os.path.join(label_dir, ann_file), 'w') as f:
            f.write('\n'.join(yolo_lines) + '\n' if yolo_lines else '')

convert_visdrone_to_yolo(TRAIN_IMG_DIR, TRAIN_ANN_DIR, TRAIN_LABEL_DIR)
convert_visdrone_to_yolo(VAL_IMG_DIR, VAL_ANN_DIR, VAL_LABEL_DIR)
print("Conversion to YOLO format complete.")

In [ ]:
import yaml

data_yaml = {
    "path": BASE_DIR,
    "train": "images/train",
    "val": "images/val",
    "nc": 10,
    "names": ["pedestrian", "people", "bicycle", "car", "van",
              "truck", "tricycle", "awning-tricycle", "bus", "motor"]
}

with open("visdrone.yaml", "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)
print("visdrone.yaml created.")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data="visdrone.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="train100",
    project="/content/drive/MyDrive/Yolov8_VisDrone_Trained",
    device="0",
    workers=4,
    patience=50,
    save=True,
    val=True
)
print("Training completed.")

In [ ]:
import os, json, time, torch
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
from sahi.predict import get_sliced_prediction
from sahi.models.ultralytics import UltralyticsDetectionModel
from ensemble_boxes import weighted_boxes_fusion
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

VAL_IMG_DIR = os.path.join(BASE_DIR, "images/val")
GT_COCO = "/content/val_gt_coco.json"
MODEL_PATH = "/content/drive/MyDrive/Yolov8_VisDrone_Trained/train100/weights/best.pt"
OUT_DIR = "/content/outputs"
os.makedirs(OUT_DIR, exist_ok=True)
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

with open(GT_COCO) as f:
    gt = json.load(f)
image_map = {img["file_name"].replace('.jpg',''): img["id"] for img in gt["images"]}

def yolo_to_coco(img_dir, label_dir, out_json):
    categories = [{"id": i, "name": str(i)} for i in range(10)]
    coco = {"images": [], "annotations": [], "categories": categories}
    ann_id = 0
    files = sorted([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))])
    for idx, fname in enumerate(files):
        label_path = os.path.join(label_dir, fname.replace('.jpg', '.txt'))
        img_path = os.path.join(img_dir, fname)
        if not os.path.exists(img_path) or not os.path.exists(label_path):
            continue
        h, w = cv2.imread(img_path).shape[:2]
        coco["images"].append({"id": idx, "file_name": fname, "height": h, "width": w})
        with open(label_path, 'r') as f:
            for line in f:
                parts = list(map(float, line.strip().split()))
                if len(parts) < 5:
                    continue
                cls, cx, cy, nw, nh = parts
                x, y = (cx - nw/2)*w, (cy - nh/2)*h
                coco["annotations"].append({
                    "id": ann_id, "image_id": idx, "category_id": int(cls),
                    "bbox": [x, y, nw*w, nh*h], "area": nw*w*nh*h, "iscrowd": 0
                })
                ann_id += 1
    with open(out_json, 'w') as f:
        json.dump(coco, f)
    return out_json

yolo_to_coco(VAL_IMG_DIR, os.path.join(BASE_DIR, "labels/val"), GT_COCO)

def compute_metrics(gt_json, pred_json, label):
    cocoGt = COCO(gt_json)
    cocoDt = cocoGt.loadRes(pred_json)
    cocoEval = COCOeval(cocoGt, cocoDt, iouType='bbox')
    cocoEval.evaluate()
    cocoEval.accumulate()
    cocoEval.summarize()
    return {
        "Method": label,
        "mAP@0.5:0.95": round(cocoEval.stats[0] * 100, 2),
        "mAP@0.5": round(cocoEval.stats[1] * 100, 2),
        "Recall@0.5": round(cocoEval.stats[8] * 100, 2)
    }

def run_inference(model_path, img_dir, out_json, label, use_sahi=False, postprocess="NMS", overlap=0.3, slice_size=640, conf_thresh=0.25):
    model = UltralyticsDetectionModel(
        model_path=model_path, confidence_threshold=conf_thresh, device=DEVICE
    )
    files = sorted([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))])
    preds, latencies = [], []

    for fname in tqdm(files, desc=label):
        base = os.path.splitext(fname)[0]
        if base not in image_map:
            continue
        img_path = os.path.join(img_dir, fname)
        h, w = cv2.imread(img_path).shape[:2]
        t0 = time.perf_counter()

        if use_sahi:
            result = get_sliced_prediction(
                img_path, model,
                slice_height=slice_size, slice_width=slice_size,
                overlap_height_ratio=overlap, overlap_width_ratio=overlap,
                postprocess_type="NMS", verbose=0
            )
            if postprocess == "WBF" and result.object_prediction_list:
                norm_boxes, scores, labels = [], [], []
                for obj in result.object_prediction_list:
                    b = obj.bbox
                    norm_boxes.append([b.minx/w, b.miny/h, b.maxx/w, b.maxy/h])
                    scores.append(obj.score.value)
                    labels.append(obj.category.id)
                fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
                    [norm_boxes], [scores], [labels], iou_thr=0.5, skip_box_thr=0.0, weights=None
                )
                for fb, fs, fl in zip(fused_boxes, fused_scores, fused_labels):
                    x1, y1, x2, y2 = fb[0]*w, fb[1]*h, fb[2]*w, fb[3]*h
                    preds.append({"image_id": int(image_map[base]), "category_id": int(fl),
                                  "bbox": [round(float(x1),2), round(float(y1),2),
                                           round(float(x2-x1),2), round(float(y2-y1),2)],
                                  "score": round(float(fs), 4)})
            else:
                for obj in result.object_prediction_list:
                    b = obj.bbox
                    preds.append({"image_id": int(image_map[base]), "category_id": int(obj.category.id),
                                  "bbox": [round(float(b.minx),2), round(float(b.miny),2),
                                           round(float(b.maxx - b.minx),2), round(float(b.maxy - b.miny),2)],
                                  "score": round(float(obj.score.value), 4)})
        else:
            res = model.model.predict(img_path, conf=conf_thresh, verbose=False, device=DEVICE)[0]
            for box in res.boxes:
                cls = int(box.cls.item())
                conf = float(box.conf.item())
                xyxy = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = xyxy
                preds.append({"image_id": int(image_map[base]), "category_id": cls,
                              "bbox": [round(float(x1),2), round(float(y1),2),
                                       round(float(x2 - x1),2), round(float(y2 - y1),2)],
                              "score": round(conf, 4)})

        dt = (time.perf_counter() - t0) * 1000
        latencies.append(dt)

    with open(out_json, 'w') as f:
        json.dump(preds, f)
    avg_lat = sum(latencies)/len(latencies) if latencies else 0
    fps = 1000/avg_lat if avg_lat > 0 else 0
    return avg_lat, fps

ablation_configs = [
    {"name": "tile640_ov0.3_conf0.25_NMS", "use_sahi": True, "postprocess": "NMS", "overlap": 0.3, "slice_size": 640, "conf": 0.25},
    {"name": "tile640_ov0.3_conf0.25_WBF", "use_sahi": True, "postprocess": "WBF", "overlap": 0.3, "slice_size": 640, "conf": 0.25},
    {"name": "tile480_ov0.3_conf0.25_NMS", "use_sahi": True, "postprocess": "NMS", "overlap": 0.3, "slice_size": 480, "conf": 0.25},
    {"name": "tile800_ov0.3_conf0.25_NMS", "use_sahi": True, "postprocess": "NMS", "overlap": 0.3, "slice_size": 800, "conf": 0.25},
    {"name": "tile640_ov0.3_conf0.15_NMS", "use_sahi": True, "postprocess": "NMS", "overlap": 0.3, "slice_size": 640, "conf": 0.15},
    {"name": "tile640_ov0.4_conf0.25_NMS", "use_sahi": True, "postprocess": "NMS", "overlap": 0.4, "slice_size": 640, "conf": 0.25}
]

results = []
for cfg in ablation_configs:
    out_json = os.path.join(OUT_DIR, f"{cfg['name']}.json")
    lat, fps = run_inference(MODEL_PATH, VAL_IMG_DIR, out_json, cfg['name'], **{k:v for k,v in cfg.items() if k != 'name'})
    metrics = compute_metrics(GT_COCO, out_json, cfg['name'])
    metrics["FPS"] = round(fps, 2)
    metrics["Avg_Latency_ms"] = round(lat, 2)
    results.append(metrics)

df_ablation = pd.DataFrame(results)
print(df_ablation[["Method", "mAP@0.5:0.95", "mAP@0.5", "Recall@0.5", "FPS"]].to_string(index=False))
df_ablation.to_csv("/content/ablation_master_results.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def get_per_class_metrics(gt_json, pred_json):
    cocoGt = COCO(gt_json)
    cocoDt = cocoGt.loadRes(pred_json)
    cocoEval = COCOeval(cocoGt, cocoDt, iouType='bbox')
    cocoEval.evaluate()
    cocoEval.accumulate()
    precisions = cocoEval.eval['precision']
    class_aps = []
    class_names = [cat['name'] for cat in cocoGt.cats.values()]
    for i in range(len(class_names)):
        p = precisions[:, :, i, 0, 2]
        p = p[p > -1]
        ap = np.mean(p) if len(p) > 0 else 0.0
        class_aps.append(ap)
    return pd.DataFrame({"Class": class_names, "AP@0.5:0.95": class_aps})

baseline_json = "/content/outputs/tile640_ov0.3_conf0.25_NMS.json"
df_baseline = get_per_class_metrics(GT_COCO, baseline_json)

plt.figure(figsize=(12, 6))
sns.barplot(x="Class", y="AP@0.5:0.95", data=df_baseline, palette="viridis")
plt.title("Per-Class AP@0.5:0.95 (Fine-tuned YOLOv8-Nano)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("/content/per_class_ap.png")
plt.show()

df_baseline.to_csv("/content/per_class_metrics_comparison.csv", index=False)
print("Per-class metrics saved.")

In [ ]:
import json, os, shutil

FINAL_PATH = "/content/drive/MyDrive/ML_Project_VisDrone_Ablation_Backup"
os.makedirs(FINAL_PATH, exist_ok=True)

backup_items = {
    "inference_outputs": ["/content/outputs"],
    "evaluation": ["/content/ablation_master_results.csv", "/content/per_class_metrics_comparison.csv"],
    "configs": ["visdrone.yaml"],
    "notebooks": ["ML_Proj_Finetuned_Yolo_ablation_per_class_analysis.ipynb"]
}

copied_count = 0
skipped_count = 0

for folder, items in backup_items.items():
    dest_folder = os.path.join(FINAL_PATH, folder)
    os.makedirs(dest_folder, exist_ok=True)
    for item in items:
        if os.path.exists(item):
            if os.path.isdir(item):
                shutil.copytree(item, os.path.join(dest_folder, os.path.basename(item)), dirs_exist_ok=True)
            else:
                shutil.copy(item, dest_folder)
            copied_count += 1
        else:
            skipped_count += 1

manifest = {
    "project": "VisDrone SOD Ablation Study",
    "backup_date": "2024-05-01",
    "items_copied": copied_count,
    "items_skipped": skipped_count
}
with open(os.path.join(FINAL_PATH, "backup_manifest.json"), 'w') as f:
    json.dump(manifest, f, indent=2)

print("Backup complete.")